<img src="https://wamugii.kento.rw/logo-emblem-128.png" width="56" align="right" alt="WAMUGII">

**Responsible AI & Deployment (Capstone)** · Week 4 · Governance, Documentation and the Capstone · Lesson 20 of 24

# Documentation: Model Cards and Datasheets

**How to use this notebook:** run the code cells **in order**, from top to bottom (press ▶ on each cell, or use **Runtime → Run all**). You can change any code and run it again. To keep your changes, use **File → Save a copy in Drive**.

The lesson, quiz and certificate are on [WAMUGII](https://wamugii.kento.rw/courses/responsible-ai-deployment).

**Learning objective:** Write a datasheet for a dataset and a model card for a model, generate them automatically from real metrics, and keep them up to date with every release.

## Why documentation?

Imagine a new developer, an auditor or a regulator in two years. They need to know: what is this model for, what data built it, how good is it, for whom does it work less well, and what must it **not** be used for? Two widely used formats answer these questions:

- **Datasheets for Datasets** (Gebru and colleagues, 2018): how a dataset was collected, what it contains, its limits and allowed uses.
- **Model Cards for Model Reporting** (Mitchell and colleagues, 2019): a short document about one model — intended use, performance broken down by group, limitations and ethical considerations.

Hugging Face shows a model card on every model page — you have seen them in Course 6.

In [ ]:
import numpy as np
import pandas as pd

def make_applicants(n=3000, seed=7, drift=0.0, concept=0.0):
    # Made-up SACCO loan applications. drift > 0: incomes, loans and late payments change.
    # concept > 0: the link to repayment changes (salaried jobs protect less, late payments hurt more).
    rng = np.random.default_rng(seed)
    gender = rng.choice(["female", "male"], n)
    area = rng.choice(["urban", "rural"], n, p=[0.6, 0.4])
    rural = area == "rural"
    income = (rng.lognormal(12.4, 0.5, n) * np.where(rural, 0.7, 1.0) * (1 - drift * 0.25)).round(-3)
    loan = (rng.uniform(50_000, 600_000, n) * (1 + drift)).round(-3)
    years = rng.integers(0, 13, n)
    late = rng.poisson(0.7 + drift, n)
    salaried = (rng.random(n) < np.where(rural, 0.3, 0.6)).astype(int)
    momo = np.clip(rng.normal(np.where(rural, 18, 45), 10), 0, None).round().astype(int)
    score = 3.0 - 1.6 * (loan / income) + 0.12 * years - 0.7 * late + 0.6 * salaried + 0.01 * momo
    score = score - concept * (1.5 * salaried + 0.6 * late - 0.04 * (momo - 30))
    repaid = (rng.random(n) < 1 / (1 + np.exp(-score))).astype(int)
    officer = score - 0.9 * rural - 0.6 * (gender == "female") + rng.normal(0, 0.6, n)
    return pd.DataFrame({
        "applicant_id": np.arange(1, n + 1), "age": rng.integers(20, 63, n), "gender": gender, "area": area,
        "income_rwf": income, "loan_rwf": loan, "years_member": years, "late_payments": late,
        "salaried": salaried, "momo_tx_month": momo,
        "approved_before": (officer > 1.2).astype(int), "repaid": repaid})

applicants = make_applicants()
print(applicants.shape, "| repaid:", round(applicants["repaid"].mean(), 3))

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

FEATURES = ["income_rwf", "loan_rwf", "years_member", "late_payments", "salaried", "momo_tx_month"]
TARGET = "repaid"
train, test = train_test_split(applicants, test_size=0.25, random_state=0, stratify=applicants[TARGET])
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
model.fit(train[FEATURES], train[TARGET])
proba = model.predict_proba(test[FEATURES])[:, 1]
print(f"accuracy {accuracy_score(test[TARGET], proba >= 0.5):.3f} | AUC {roc_auc_score(test[TARGET], proba):.3f}")

## A datasheet, generated from the data

In [ ]:
def datasheet(df, name):
    lines = [f"# Datasheet: {name}", "",
             "## Motivation", "Created to train and audit a loan-repayment model for a SACCO (practice data, no real people).", "",
             "## Composition", f"- Rows: {len(df):,}", f"- Columns: {len(df.columns)}"]
    for col in df.columns:
        s = df[col]
        if pd.api.types.is_numeric_dtype(s):
            info = f"min {s.min():,.0f}, median {s.median():,.0f}, max {s.max():,.0f}"
        else:
            info = ", ".join(f"{k} {v:.0%}" for k, v in s.value_counts(normalize=True).items())
        lines.append(f"- `{col}`: {info}; missing {s.isna().mean():.0%}")
    lines += ["", "## Known issues",
              "- `approved_before` records past officer decisions that were unfair to rural applicants and women.",
              "- Rural applicants have lower incomes and fewer salaried jobs; several features act as proxies for area.",
              "", "## Allowed uses", "Training and auditing credit-risk models with human review. Not for marketing or for decisions without review.",
              "", "## Personal data", "Contains no names or contact details. Gender, area and age are kept for fairness audits only."]
    return "\n".join(lines)

sheet = datasheet(applicants, "SACCO loan applications v1")
open("DATASHEET.md", "w").write(sheet)
print(sheet)

## A model card, generated from the metrics

In [ ]:
def group_metrics(data, group, approved):
    out = []
    for name, g in data.assign(approved=approved).groupby(group):
        out.append(f"| {group} = {name} | {len(g)} | {g['approved'].mean():.1%} | "
                   f"{g.loc[g[TARGET] == 1, 'approved'].mean():.1%} | {roc_auc_score(g[TARGET], g['score']):.3f} |")
    return out

scored = test.assign(score=model.predict_proba(test[FEATURES])[:, 1])
approved = (scored["score"] >= 0.5).astype(int)

card = "\n".join([
    "# Model card: loan-repayment v1.0.0", "",
    "## Intended use",
    "- Supports SACCO loan officers by estimating the probability that an applicant repays.",
    "- Output: approve, or refer to an officer. **The model never refuses a loan on its own.**", "",
    "## Out-of-scope uses",
    "- Fully automatic refusals; marketing; employment or insurance decisions; applicants outside the SACCO.", "",
    "## Model", f"- Logistic regression on standardised features: {', '.join(FEATURES)}.",
    "- Gender, area and age are NOT inputs.", "",
    "## Data", "- Trained on 2,250 applications; tested on 750. See DATASHEET.md.", "",
    "## Performance (test set)",
    f"- AUC {roc_auc_score(test[TARGET], scored['score']):.3f}, accuracy {accuracy_score(test[TARGET], approved):.3f}", "",
    "## Performance by group",
    "| Group | People | Approval rate | Approved if would repay | AUC |", "|---|---|---|---|---|",
    *group_metrics(scored, "area", approved), *group_metrics(scored, "gender", approved), "",
    "## Limitations and ethical considerations",
    "- Rural applicants are approved less often (disparate impact below 0.8); human review of refused rural applicants is required.",
    "- Performance will fall under concept drift; outcomes are monitored monthly.",
    "- Mobile-money activity can be inflated by applicants; its effect is monitored.", "",
    "## Owners", "- System owner: SACCO credit manager. Developer: data team. Next review: in 3 months.",
])
open("MODEL_CARD.md", "w").write(card)
print(card)

Because the card is **generated by code** from the real metrics, it cannot become out of date: regenerate it in CI for every release (Lesson 7) and store it next to the model in the registry (Lesson 8).

## What makes good documentation

- **Honest:** includes weaknesses and groups where the model works less well.
- **Specific:** numbers, dates, versions, owners — not "the model is fair".
- **For its readers:** a short summary for managers, details for auditors.
- **Versioned:** one card per model version.

## Key terms

- **Datasheet** — documentation of a dataset's origin, contents, limits and uses.
- **Model card** — documentation of a model's use, performance by group, limits and ethics.
- **Intended use / out-of-scope use** — what the model is and is not for.
- **Disaggregated metrics** — performance reported separately for each group.

**Takeaway:** Document datasets with datasheets and models with model cards, generate them from real metrics in every release, and be honest about limits.

## Try it yourself

1. Add an "age band" section to the model card's group table.
2. Add a "Change log" section listing what changed since the previous version.
3. Write a five-sentence summary of the model card for the SACCO board.
4. Find a model card on Hugging Face and list which of our sections it has and which it misses.

---
Finished? Go back to [Responsible AI & Deployment (Capstone) on WAMUGII](https://wamugii.kento.rw/courses/responsible-ai-deployment), tick the lesson as done and take the quiz.

*© WAMUGII TECH SOLUTIONS · Original course material for WAMUGII AI Program students.*